# Feature-Target-Übersicht

Dieses Notebook zeigt automatisch die Zusammenhänge aller Features mit einer gewählten Zielvariable.

- Numerische Features → Scatterplots
- Kategoriale Features → Boxplots
- Die Darstellungen zeigen **marginale Zusammenhänge** und berücksichtigen die übrigen Features noch nicht.


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'Datensatz: {filename}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
display(df.head())


In [ ]:
print('Verfügbare Spalten:')
for col in df.columns:
    print('-', col)


## Zielvariable festlegen
Passe `TARGET` an deinen Datensatz an.

In [ ]:
TARGET = 'marketvalue'   # <- anpassen

assert TARGET in df.columns, f"Zielvariable '{TARGET}' nicht gefunden."
features = [col for col in df.columns if col != TARGET]
numeric_features = df[features].select_dtypes(include='number').columns.tolist()
categorical_features = df[features].select_dtypes(exclude='number').columns.tolist()

print('Zielvariable:', TARGET)
print('Numerische Features:', numeric_features)
print('Kategoriale Features:', categorical_features)


## Numerische Features gegen die Zielvariable

In [ ]:
import math
import matplotlib.pyplot as plt

if len(numeric_features) == 0:
    print('Keine numerischen Features gefunden.')
else:
    n_cols = 3
    n_rows = math.ceil(len(numeric_features) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = [axes] if len(numeric_features) == 1 else axes.flatten()

    for i, feature in enumerate(numeric_features):
        plot_df = df[[feature, TARGET]].dropna()
        axes[i].scatter(plot_df[feature], plot_df[TARGET], alpha=0.5)
        axes[i].set_title(f'{TARGET} vs. {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel(TARGET)
        axes[i].grid(alpha=0.2)

    for j in range(len(numeric_features), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


## Kategoriale Features gegen die Zielvariable
Features mit mehr als 15 Kategorien werden übersprungen.

In [ ]:
if len(categorical_features) == 0:
    print('Keine kategorialen Features gefunden.')
else:
    for feature in categorical_features:
        n_cat = df[feature].nunique(dropna=True)
        if n_cat <= 15:
            plot_df = df[[feature, TARGET]].dropna()
            plt.figure(figsize=(10,5))
            plot_df.boxplot(column=TARGET, by=feature, grid=False, rot=45)
            plt.title(f'{TARGET} nach {feature}')
            plt.suptitle('')
            plt.xlabel(feature)
            plt.ylabel(TARGET)
            plt.tight_layout()
            plt.show()
        else:
            print(f'Übersprungen: {feature} ({n_cat} Kategorien)')


## Leitfragen

1. Welche Features zeigen einen klaren Zusammenhang mit der Zielvariable?
2. Ist der Zusammenhang eher linear, quadratisch, polynomial, periodisch oder kaum erkennbar?
3. Welche kategorialen Features zeigen Unterschiede zwischen Gruppen?
4. Welche Hypothesenräume erscheinen für die einzelnen Features plausibel?
